# Actividad 02: Normalización (1NF, 2NF, 3NF)

Parasol Corp tiene una única tabla `orders_raw` que lo almacena todo en un solo lugar. Esto provoca redundancia y anomalías.

**Tu trabajo:** normalizar esta tabla paso a paso hasta alcanzar la **Tercera Forma Normal (3NF)**.

In [1]:
import sys

print("Python utilizado por Jupyter:")
print(sys.executable)

print("\nEntorno virtual:")
print(sys.prefix)

print("\nPython base:")
print(sys.base_prefix)

Python utilizado por Jupyter:
c:\Users\anto-\OneDrive\Desktop\PONTIA\Tema 4 - Modelado de Datos y Data Warehousing\.venv\Scripts\python.exe

Entorno virtual:
c:\Users\anto-\OneDrive\Desktop\PONTIA\Tema 4 - Modelado de Datos y Data Warehousing\.venv

Python base:
C:\Users\anto-\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none


## El camino de la normalización

Vas a pasar de una tabla con todo mezclado a un modelo en **3NF**, paso a paso.

```mermaid
flowchart LR
    A["orders_raw<br/>1 tabla con todo mezclado"] --> B["1NF<br/>orders_1nf"]
    B --> C["2NF<br/>orders + products + order_items"]
    C --> D["3NF<br/>customers + orders_3nf + order_items + products"]
```


In [2]:
import duckdb
import numpy

con = duckdb.connect(':memory:')

# Crea la tabla desnormalizada
con.execute("""
    CREATE TABLE orders_raw (
        order_id INTEGER,
        customer_name VARCHAR,
        customer_city VARCHAR,
        product_name VARCHAR,
        quantity INTEGER,
        product_category VARCHAR
    );
""")

con.execute("""
    INSERT INTO orders_raw VALUES
    (1, 'Alice Garcia', 'Madrid', 'Laptop', 1, 'Electronics'),
    (1, 'Alice Garcia', 'Madrid', 'Mouse', 2, 'Electronics'),
    (2, 'Bob Smith', 'London', 'Keyboard', 1, 'Electronics'),
    (2, 'Bob Smith', 'London', 'Monitor', 1, 'Electronics'),
    (3, 'Alice Garcia', 'Madrid', 'Headphones', 1, 'Accessories'),
    (4, 'Carol White', 'Paris', 'Laptop', 2, 'Electronics'),
    (4, 'Carol White', 'Paris', 'USB Cable', 3, 'Accessories');
""")

print('Denormalized table:')
# Mostrar los registros originales
con.execute("""
    SELECT *
    FROM orders_raw
    ORDER BY order_id, product_name;
""").df()

ModuleNotFoundError: No module named 'numpy'

In [ ]:
import duckdb

con = duckdb.connect(':memory:')

# Crea la tabla desnormalizada
con.execute("""
    CREATE TABLE orders_raw (
        order_id INTEGER,
        customer_name VARCHAR,
        customer_city VARCHAR,
        product_name VARCHAR,
        quantity INTEGER,
        product_category VARCHAR
    );
""")

con.execute("""
    INSERT INTO orders_raw VALUES
    (1, 'Alice Garcia', 'Madrid', 'Laptop', 1, 'Electronics'),
    (1, 'Alice Garcia', 'Madrid', 'Mouse', 2, 'Electronics'),
    (2, 'Bob Smith', 'London', 'Keyboard', 1, 'Electronics'),
    (2, 'Bob Smith', 'London', 'Monitor', 1, 'Electronics'),
    (3, 'Alice Garcia', 'Madrid', 'Headphones', 1, 'Accessories'),
    (4, 'Carol White', 'Paris', 'Laptop', 2, 'Electronics'),
    (4, 'Carol White', 'Paris', 'USB Cable', 3, 'Accessories');
""")

print('Denormalized table:')
# Mostrar los registros originales
con.execute("""
    SELECT *
    FROM orders_raw
    ORDER BY order_id, product_name;
""").df()

Denormalized table:


ModuleNotFoundError: No module named 'numpy'

### Observa los problemas

- `'Alice Garcia', 'Madrid'` aparece **3 veces** -> redundancia
- Si Alice se muda a Barcelona, necesitamos actualizar **3 filas** -> anomalía de actualizacion
- Si borramos el pedido 3, perdemos el hecho de que Alice le gustan los auriculares -> anomalía de borrado
- No podemos anadir un nuevo cliente sin un pedido -> anomalía de insercion

### Las dependencias que causan los problemas

`customer_city` depende de `customer_name`, no de la PK: es una **dependencia transitiva**. `customer_name` y `product_category` dependen solo de una parte de la PK: son **dependencias parciales**.

```mermaid
flowchart LR
    subgraph pk["PK compuesta: order_id + product_name"]
        O[order_id]
        P[product_name]
    end
    O -->|dependencia parcial| CN[customer_name]
    CN -->|dependencia transitiva| CC[customer_city]
    P -->|dependencia parcial| PC[product_category]
    O --> Q[quantity]
    P --> Q
```


---

## Paso 1: Primera Forma Normal (1NF)

> **Recuerda:** la **1NF** exige valores atómicos, filas únicas (con PK) y ningún grupo repetido.

Nuestra tabla ya tiene valores atómicos. Pero necesitamos una PK adecuada.

**Tarea:** Crea `orders_1nf` con una **Primary Key compuesta** `(order_id, product_name)`.

### Así queda `orders_1nf`

Una fila por cada par `(order_id, product_name)`, con **PK compuesta**.

```mermaid
erDiagram
    ORDERS_1NF {
        int order_id PK
        string product_name PK
        string customer_name
        string customer_city
        int quantity
        string product_category
    }
```


In [ ]:
# TODO: Crea la tabla orders_1nf



---

## Paso 2: Segunda Forma Normal (2NF)

> **Recuerda:** la **2NF** exige estar en 1NF y que cada columna no clave dependa de **toda** la PK (sin dependencias parciales).

### El problema

Nuestra PK es `(order_id, product_name)`. Pero:

- `customer_name` depende solo de `order_id` -> **dependencia parcial**
- `customer_city` depende solo de `order_id` -> **dependencia parcial**
- `product_category` depende solo de `product_name` -> **dependencia parcial**

### Enfoque: dividir en 3 tablas

```
orders (order_id PK, customer_name, customer_city)
products (product_name PK, product_category)
order_items (order_id FK, product_name FK, quantity)
```

### Así queda el modelo 2NF

Las dependencias parciales se van a sus propias tablas. `quantity` depende de la PK completa, así que se queda en `order_items`.

```mermaid
erDiagram
    ORDERS ||--o{ ORDER_ITEMS : FK-order
    PRODUCTS ||--o{ ORDER_ITEMS : FK-product
    ORDERS {
        int order_id PK
        string customer_name
        string customer_city
    }
    PRODUCTS {
        string product_name PK
        string product_category
    }
    ORDER_ITEMS {
        int order_id PK
        string product_name PK
        int quantity
    }
```


In [ ]:
# TODO: Crea la tabla orders



In [ ]:
# TODO: Crea la tabla products



In [ ]:
# TODO: Crea la tabla order_items (tabla intermedia)



### Verificación - 2NF

Ejecuta esto para comprobar tus tablas 2NF:

In [ ]:
# Descomenta para verificar 2NF
# print('=== orders ===')
# con.execute('SELECT * FROM orders').fetchdf()
# print('=== products ===')
# con.execute('SELECT * FROM products').fetchdf()
# print('=== order_items ===')
# con.execute('SELECT * FROM order_items').fetchdf()


---

## Paso 3: Tercera Forma Normal (3NF)

> **Recuerda:** la **3NF** exige estar en 2NF y que ninguna columna no clave dependa de otra columna no clave (sin dependencias transitivas).

### El problema

En nuestra tabla `orders`:

```
order_id (PK) -> customer_name -> customer_city
```

`customer_city` depende de `customer_name`, que depende de `order_id`. Esta es una **dependencia transitiva**.

### Enfoque: extraer una tabla de clientes

```
customers (customer_id PK, customer_name, customer_city)
orders_3nf (order_id PK, customer_id FK)
```

### Así queda el modelo 3NF

Los clientes pasan a su propia tabla y `orders_3nf` los enlaza con una FK. Este es el modelo final.

```mermaid
erDiagram
    CUSTOMERS ||--o{ ORDERS_3NF : FK-customer
    ORDERS_3NF ||--o{ ORDER_ITEMS : FK-order
    PRODUCTS ||--o{ ORDER_ITEMS : FK-product
    CUSTOMERS {
        int customer_id PK
        string customer_name
        string customer_city
    }
    ORDERS_3NF {
        int order_id PK
        int customer_id FK
    }
    ORDER_ITEMS {
        int order_id PK
        string product_name PK
        int quantity
    }
    PRODUCTS {
        string product_name PK
        string product_category
    }
```


In [ ]:
# TODO: Crea la tabla customers



In [ ]:
# TODO: Crea la tabla orders_3nf (con customer_id FK en vez de customer_name)



### Verificación - 3NF

Ejecuta un JOIN para reconstruir los datos originales desde las tablas normalizadas:

In [ ]:
# Descomenta para verificar 3NF
# result = con.execute("""
#     SELECT o.order_id, c.customer_name, c.customer_city,
#            oi.product_name, oi.quantity, p.product_category
#     FROM orders_3nf o
#     JOIN customers c ON o.customer_id = c.customer_id
#     JOIN order_items oi ON o.order_id = oi.order_id
#     JOIN products p ON oi.product_name = p.product_name
#     ORDER BY o.order_id
# """).fetchall()
# for row in result:
#     print(row)


---

## Guarda tu trabajo

Copia todas tus sentencias SQL en `scripts/normalization.py`.

El fichero debe:
1. Crear la tabla original `orders_raw` con sus 7 filas de datos
2. Crear todas las tablas normalizadas (1NF, 2NF, 3NF) y poblarlas

```bash
pytest tests/ -v
```